In [ ]:
!pip install mlxtend openpyxl -q
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
print("Click 'Choose Files' and upload online_retail_II.xlsx")
uploaded = files.upload()

In [ ]:
import io, pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Load Data
df = pd.read_excel(io.BytesIO(uploaded['online_retail_II.xlsx']), sheet_name='Year 2010-2011')

# Clean Data
df = df[df['Customer ID'].notnull()]
df = df[~df['Invoice'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
print("Data Cleaned. Total Bills:", df['Invoice'].nunique())

# Create Baskets
basket = df.groupby('Invoice')['Description'].apply(list)

# Run Apriori
te = TransactionEncoder()
df_encoded = pd.DataFrame(te.fit(basket).transform(basket), columns=te.columns_)
frequent_itemsets = apriori(df_encoded, min_support=0.01, use_colnames=True) # 1% support
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
rules = rules[rules['confidence'] > 0.3] # 30% confidence
rules = rules.sort_values('lift', ascending=False).head(50)

# Clean for Excel/PowerBI
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

Data Cleaned. Total Bills: 18536


,antecedents,consequents,support,confidence,lift
536,REGENCY TEA PLATE GREEN,REGENCY TEA PLATE PINK,0.010898,0.748148,61.909259
537,REGENCY TEA PLATE PINK,REGENCY TEA PLATE GREEN,0.010898,0.901786,61.909259
901,"POPPY'S PLAYHOUSE BEDROOM , POPPY'S PLAYHOUSE ...",POPPY'S PLAYHOUSE LIVINGROOM,0.010035,0.732283,53.863517
904,POPPY'S PLAYHOUSE LIVINGROOM,"POPPY'S PLAYHOUSE BEDROOM , POPPY'S PLAYHOUSE ...",0.010035,0.738095,53.863517
531,REGENCY SUGAR BOWL GREEN,REGENCY MILK JUG PINK,0.011114,0.768657,52.381694
530,REGENCY MILK JUG PINK,REGENCY SUGAR BOWL GREEN,0.011114,0.757353,52.381694
903,POPPY'S PLAYHOUSE BEDROOM,"POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE L...",0.010035,0.588608,50.746188
902,"POPPY'S PLAYHOUSE KITCHEN, POPPY'S PLAYHOUSE L...",POPPY'S PLAYHOUSE BEDROOM,0.010035,0.865116,50.746188
540,REGENCY TEA PLATE PINK,REGENCY TEA PLATE ROSES,0.010628,0.879464,49.700457
541,REGENCY TEA PLATE ROSES,REGENCY TEA PLATE PINK,0.010628,0.600610,49.700457


In [ ]:
rules.to_csv('mba_rules.csv', index=False)
files.download('mba_rules.csv')
print("File Downloaded! Check 'Downloads' folder")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File Downloaded! Check 'Downloads' folder
